# Project 03: Credit Scoring with Machine Learning

**Objective:** Build a probability-of-default (PD) model combining classical scorecard
methodology (WoE/IV) with modern ML (XGBoost) and post-hoc explainability (SHAP).

**Pipeline:**
1. Load synthetic credit data and explore distributions
2. Compute Information Value (IV) for feature selection
3. Fit logistic regression with WoE-encoded features
4. Fit XGBoost on raw features
5. Compare: ROC, KS, calibration curves
6. SHAP analysis (global + local)
7. Generate SR 11-7 model card

In [ ]:
import sys
from pathlib import Path

# Add project and shared library paths
PROJECT_ROOT = Path.cwd().parent
REPO_ROOT = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "src"))

import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd
import yaml

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Load Configuration and Data

In [ ]:
# Load config
config_path = PROJECT_ROOT / "configs" / "default.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)
print("Config loaded:")
config

In [ ]:
from data import generate_synthetic_credit_data, preprocess, train_test_split_temporal

# Generate synthetic credit data
raw_df = generate_synthetic_credit_data(
    n_samples=config["data"]["n_samples"],
    seed=config["data"]["seed"],
)
print(f"Dataset shape: {raw_df.shape}")
print(f"Default rate: {raw_df['default'].mean():.2%}")
raw_df.head()

## 2. Exploratory Data Analysis

In [ ]:
raw_df.describe()

In [ ]:
# Feature distributions: defaults vs non-defaults
numeric_feats = ["income", "age", "employment_length", "loan_amount",
                 "interest_rate", "dti", "credit_history_length", "n_delinquencies"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, feat in zip(axes.ravel(), numeric_feats):
    for label, group in raw_df.groupby("default"):
        ax.hist(group[feat], bins=30, alpha=0.5,
                label=f"Default={label}", density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)
fig.suptitle("Feature Distributions by Default Status", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Loan purpose breakdown
purpose_default = raw_df.groupby("loan_purpose")["default"].agg(["mean", "count"])
purpose_default.columns = ["default_rate", "count"]
purpose_default.sort_values("default_rate", ascending=False)

## 3. Preprocessing and Temporal Split

In [ ]:
# Preprocess: handle NaN, one-hot encode loan_purpose
df = preprocess(raw_df)
print(f"Preprocessed shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Temporal split
train_df, test_df = train_test_split_temporal(df, test_ratio=config["data"]["test_ratio"])
print(f"\nTrain: {len(train_df)} rows, Test: {len(test_df)} rows")
print(f"Train default rate: {train_df['default'].mean():.2%}")
print(f"Test default rate:  {test_df['default'].mean():.2%}")
print(f"Train date range: {train_df['application_date'].min()} to {train_df['application_date'].max()}")
print(f"Test date range:  {test_df['application_date'].min()} to {test_df['application_date'].max()}")

In [ ]:
# Define feature columns (exclude target and date)
exclude = {"default", "application_date"}
feature_cols = [c for c in df.columns if c not in exclude]

X_train = train_df[feature_cols]
y_train = train_df["default"]
X_test = test_df[feature_cols]
y_test = test_df["default"]

print(f"Features ({len(feature_cols)}): {feature_cols}")

## 4. Information Value (IV) Feature Screening

IV interpretation (Siddiqi, 2017):
- < 0.02: Not predictive
- 0.02-0.1: Weak
- 0.1-0.3: Medium
- 0.3-0.5: Strong
- > 0.5: Suspicious

In [ ]:
from risk_analyst.models.credit import compute_all_iv

iv_df = compute_all_iv(X_train, y_train, n_bins=config["features"]["woe_bins"])

# Color-code by IV strength
def iv_category(val):
    if val < 0.02:
        return "Not predictive"
    elif val < 0.1:
        return "Weak"
    elif val < 0.3:
        return "Medium"
    elif val < 0.5:
        return "Strong"
    else:
        return "Suspicious"

iv_df["strength"] = iv_df["iv"].apply(iv_category)
print("Information Value by Feature:")
print(iv_df.to_string(index=False))

In [ ]:
# IV bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#d32f2f" if v >= 0.3 else "#1976d2" if v >= 0.1
          else "#ffa000" if v >= 0.02 else "#9e9e9e" for v in iv_df["iv"]]
ax.barh(iv_df["feature"], iv_df["iv"], color=colors)
ax.axvline(x=0.02, color="gray", linestyle="--", alpha=0.7, label="Min IV threshold")
ax.set_xlabel("Information Value")
ax.set_title("Feature Information Value (IV)")
ax.legend()
fig.tight_layout()
plt.show()

## 5. Model Fitting

### 5a. Logistic Regression with WoE

In [ ]:
from model import CreditScoreModel

# Fit logistic regression (WoE-based scorecard)
lr_model = CreditScoreModel(config)
lr_model.fit_logistic(X_train, y_train)

lr_proba_test = lr_model.predict_proba(X_test)
print("Logistic Regression fitted.")
print(f"  Selected features (IV >= {config['features']['min_iv']}): {lr_model._selected_features}")
print(f"  Coefficients: {dict(zip(lr_model._feature_names, lr_model._logistic.coef_[0].round(4)))}")

### 5b. XGBoost

In [ ]:
# Fit XGBoost on raw features
xgb_model = CreditScoreModel(config)
xgb_model.fit_xgboost(X_train, y_train)

xgb_proba_test = xgb_model.predict_proba(X_test)
print("XGBoost fitted.")
print(f"  n_estimators: {config['xgboost']['n_estimators']}")
print(f"  max_depth: {config['xgboost']['max_depth']}")

## 6. Model Comparison

### 6a. ROC Curves

In [ ]:
from evaluate import compute_gini, compute_ks_statistic
from sklearn.metrics import brier_score_loss, roc_auc_score, roc_curve

# Metrics
models = {
    "Logistic (WoE)": lr_proba_test,
    "XGBoost": xgb_proba_test,
}

print(f"{'Model':<18} {'AUC':>8} {'KS':>8} {'Gini':>8} {'Brier':>8}")
print("-" * 52)
for name, proba in models.items():
    auc = roc_auc_score(y_test, proba)
    ks = compute_ks_statistic(y_test.values, proba)
    gini = compute_gini(y_test.values, proba)
    brier = brier_score_loss(y_test, proba)
    print(f"{name:<18} {auc:>8.4f} {ks:>8.4f} {gini:>8.4f} {brier:>8.4f}")

In [ ]:
# Side-by-side ROC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, proba) in zip(axes, models.items()):
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, linewidth=2, label=f"AUC = {auc:.4f}")
    ax.plot([0, 1], [0, 1], "k--", linewidth=1)
    ax.set_xlabel("FPR")
    ax.set_ylabel("TPR")
    ax.set_title(f"ROC: {name}")
    ax.legend(loc="lower right")
    ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

### 6b. KS Charts

In [ ]:
from evaluate import plot_ks_chart

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, proba) in zip(axes, models.items()):
    plot_ks_chart(y_test.values, proba, ax=ax)
    ax.set_title(f"KS: {name}")
fig.tight_layout()
plt.show()

### 6c. Calibration Curves

In [ ]:
from evaluate import plot_calibration_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, proba) in zip(axes, models.items()):
    plot_calibration_curve(y_test.values, proba, n_bins=10, ax=ax)
    ax.set_title(f"Calibration: {name}")
fig.tight_layout()
plt.show()

### 6d. Probability Calibration (Isotonic Regression)

In [ ]:
# Calibrate XGBoost using half the test set, evaluate on the other half
n_half = len(X_test) // 2
X_cal, y_cal = X_test.iloc[:n_half], y_test.iloc[:n_half]
X_eval, y_eval = X_test.iloc[n_half:], y_test.iloc[n_half:]

xgb_model.calibrate(X_cal, y_cal, method=config["calibration"]["method"])
cal_proba = xgb_model.predict_proba(X_eval)

# Compare raw vs calibrated
raw_proba_eval = xgb_proba_test[n_half:]

print(f"Raw XGBoost Brier:       {brier_score_loss(y_eval, raw_proba_eval):.4f}")
print(f"Calibrated XGBoost Brier: {brier_score_loss(y_eval, cal_proba):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_calibration_curve(y_eval.values, raw_proba_eval, n_bins=10, ax=axes[0])
axes[0].set_title("Before Calibration")
plot_calibration_curve(y_eval.values, cal_proba, n_bins=10, ax=axes[1])
axes[1].set_title("After Isotonic Calibration")
fig.tight_layout()
plt.show()

## 7. SHAP Explainability

### 7a. Global Feature Importance

In [ ]:
import shap

# Use a fresh XGBoost model (without calibration wrapper) for SHAP
xgb_explain = CreditScoreModel(config)
xgb_explain.fit_xgboost(X_train, y_train)

# Global SHAP summary
summary_df = xgb_explain.explain(X_test)
print("SHAP Global Feature Importance:")
print(summary_df.to_string(index=False))

In [ ]:
# SHAP beeswarm plot (using shap library directly for richer viz)
explainer = shap.TreeExplainer(xgb_explain._xgboost)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title("SHAP Beeswarm Plot (XGBoost)")
plt.tight_layout()
plt.show()

### 7b. Local Explanation (Single Prediction)

In [ ]:
# Pick a defaulter from the test set for local explanation
default_indices = y_test[y_test == 1].index.tolist()
example_idx_global = default_indices[0]
example_idx_local = y_test.index.get_loc(example_idx_global)

print(f"Explaining observation at test index {example_idx_local} (actual default={y_test.iloc[example_idx_local]})")
print(f"Predicted PD: {xgb_explain.predict_proba(X_test)[example_idx_local]:.4f}")
print("\nFeature values:")
print(X_test.iloc[example_idx_local])

In [ ]:
# SHAP waterfall for this prediction
local_explanation = xgb_explain.explain(X_test, idx=example_idx_local)

# Display as table
local_df = pd.DataFrame({
    "Feature": local_explanation["feature_names"],
    "Value": local_explanation["feature_values"],
    "SHAP": local_explanation["shap_values"],
}).sort_values("SHAP", key=abs, ascending=False)

print(f"Base value (E[f(x)]): {local_explanation['base_value']:.4f}")
print(local_df.to_string(index=False))

In [ ]:
# SHAP waterfall plot
shap_explanation = shap.Explanation(
    values=shap_values[example_idx_local],
    base_values=explainer.expected_value,
    data=X_test.iloc[example_idx_local].values,
    feature_names=X_test.columns.tolist(),
)
plt.figure(figsize=(10, 6))
shap.waterfall_plot(shap_explanation, show=False)
plt.title(f"SHAP Waterfall -- Observation {example_idx_local}")
plt.tight_layout()
plt.show()

## 8. Model Card (SR 11-7)

Generate a structured model card following the Fed's SR 11-7 model risk management guidance.

In [ ]:
from evaluate import model_card
from IPython.display import Markdown

# Compute final metrics for the card
xgb_final_proba = xgb_explain.predict_proba(X_test)
final_metrics = {
    "auc": roc_auc_score(y_test, xgb_final_proba),
    "ks": compute_ks_statistic(y_test.values, xgb_final_proba),
    "gini": compute_gini(y_test.values, xgb_final_proba),
    "brier": brier_score_loss(y_test, xgb_final_proba),
}

card = model_card(
    model_name="XGBoost Credit Scorecard v1.0",
    metrics=final_metrics,
    features=feature_cols,
    limitations=[
        "Trained on synthetic data -- real-world performance may differ significantly.",
        "No macro-economic features (unemployment, GDP) included.",
        "Single-period snapshot; no account-level panel dynamics.",
        "Loan purpose one-hot encoding may not capture interaction effects.",
        "Class imbalance handled via scale_pos_weight only -- no SMOTE or undersampling.",
        "No adverse action reason code ordering (ECOA/Reg B compliance requires further work).",
    ],
)

Markdown(card)

In [ ]:
# Save the model card
card_path = PROJECT_ROOT / "results" / "model_card.md"
card_path.write_text(card)
print(f"Model card saved to {card_path}")

## Summary

**Key findings:**
- IV analysis identifies the most predictive features for default (typically DTI, delinquencies, interest rate)
- XGBoost achieves higher AUC than the WoE-based logistic regression
- Isotonic calibration improves probability reliability
- SHAP provides both global importance rankings and local prediction explanations
- The model card documents the model per SR 11-7 requirements

**Next steps:**
- Train on real Lending Club data
- Add macro features from FRED
- Implement Bayesian hyperparameter tuning
- Add reject inference for selection bias correction